# Daniel Rojo Mata
# Aprendizaje Automatizado, Tarea 2

# **PROBLEMA 4: Regresión logística vs clasificador bayesiano ingenuo**

Compara los métodos de regresión logística y el clasificador bayesiano ingenuo en las siguientes tareas:

- Clasificación de spam  
- Clasificación de tumores de seno

Discute qué modelo seleccionarías y por qué. Reporta el desempeño del modelo seleccionado en el conjunto de prueba. Todos los modelos deberán ser evaluados con 10 repeticiones de validación cruzada estratificada de 5 particiones.

# Se importan las librerías

In [1]:
# --- Manejo de datos y cálculo numérico ---
import pandas as pd  # Para manejo de datos en estructuras tipo DataFrame
import numpy as np   # Para operaciones numéricas y manejo de arrays

# --- Visualización ---
import matplotlib.pyplot as plt  # Para generar gráficas (líneas, barras, etc.)

# --- Preprocesamiento ---
from sklearn.preprocessing import StandardScaler  # Para normalizar características (media 0, varianza 1)
from sklearn.impute import SimpleImputer         # Para imputar valores faltantes (ej. reemplazar NaNs con la media)

# --- Validación cruzada ---
from sklearn.model_selection import StratifiedKFold           # Validación cruzada manteniendo proporciones de clase
from sklearn.model_selection import RepeatedStratifiedKFold   # Validación cruzada repetida (ej. 10x5 folds)
from sklearn.model_selection import cross_val_score           # Para evaluar modelos con validación cruzada

# --- Modelos ---
from sklearn.linear_model import LogisticRegression  # Modelo de regresión logística (supervisado, lineal)
from sklearn.naive_bayes import GaussianNB           # Clasificador bayesiano ingenuo basado en distribución Gaussiana

# --- Evaluación ---
from sklearn.metrics import accuracy_score  # Para medir la precisión (accuracy) de un modelo

# Se importa la data

In [2]:
# Archivos con los que se trabajará
# Favor de tener los archivos en la misma carpeta en donde se encuentra este documento
spam = "spam.csv"
cancer = "cancer.csv"

In [3]:
# Cargar el dataset de spam
# - 'spam' es la ruta o variable con el archivo CSV
# - header=None indica que el archivo no tiene nombres de columna
# - sep=None y engine="python" permiten que pandas detecte automáticamente el separador
data_spam = pd.read_csv(spam, 
                        header=None,
                        sep=None,
                        engine="python")

# Cargar el dataset de cáncer
# - También sin encabezados, por eso se usa header=None
data_cancer = pd.read_csv(cancer, 
                          header=None)

## Se limpia la data

In [4]:
# Se verifica si hay "?" o NaN en los data frames
def cosos_raros(data, string):
    # Verificar si existe al menos un "?"
    print(f"¿Hay '?' en el DataFrame {string}?:", (data == "?").any().any())
    # Verificar si existe al menos un NaN
    print(f"¿Hay NaN en el DataFrame {string}?:", data.isna().any().any())

cosos_raros(data_cancer, cancer), cosos_raros(data_spam, spam)

¿Hay '?' en el DataFrame cancer.csv?: True
¿Hay NaN en el DataFrame cancer.csv?: False
¿Hay '?' en el DataFrame spam.csv?: False
¿Hay NaN en el DataFrame spam.csv?: False


(None, None)

## Se limpia: data_cancer

In [5]:
# Primero aseguramos que las columnas sean numéricas (importante si estaban como strings por los "?")
data_cancer = data_cancer.apply(pd.to_numeric, errors='coerce')

# Imputar con la media
imputer = SimpleImputer(strategy="mean")
data_cancer_imputed = pd.DataFrame(imputer.fit_transform(data_cancer), columns=data_cancer.columns)

# Verificamos que no haya NaNs
print("¿Hay NaN después de imputar?:", data_cancer_imputed.isna().any().any())

¿Hay NaN después de imputar?: False


## Se procesa la data

In [6]:
# Se eliminan las etiquetas (última columna) para obtener las características (X)
X_cancer = data_cancer_imputed.drop(columns=data_cancer_imputed.columns[-1])

# Se extrae la última columna como variable objetivo (y)
y_cancer = data_cancer_imputed.iloc[:, -1]

# Se obtiene la última columna como la etiqueta (y)
y_spam = data_spam.iloc[:, -1]

# Se eliminan las etiquetas para obtener las características (todas las columnas excepto la última)
X_spam = data_spam.drop(columns=data_spam.columns[-1])

# --- Escalado de datos ---

# Normalizar características del dataset de cáncer
scaler_cancer = StandardScaler()
X_cancer_scaled = scaler_cancer.fit_transform(X_cancer)

# Normalizar características del dataset de spam
scaler_spam = StandardScaler()
X_spam_scaled = scaler_spam.fit_transform(X_spam)

# **Se evalúan los modelos (de Sklearn)**

In [7]:
# --- Función para evaluar modelos de sklearn ---
def evaluar_modelo(X, y, nombre_dataset):
    """
    Evalúa dos modelos de clasificación (Regresión Logística y Naive Bayes)
    utilizando validación cruzada estratificada repetida.

    Parámetros:
    -----------
    X : array-like, shape (n_samples, n_features)
        Matriz de características (atributos de entrada).

    y : array-like, shape (n_samples,)
        Vector de etiquetas (clases).

    nombre_dataset : str
        Nombre del conjunto de datos, usado solo para impresión de resultados.

    Salida:
    -------
    Imprime en consola el accuracy promedio y la desviación estándar para cada modelo.
    """

    print(f"Evaluación en dataset: {nombre_dataset}")
    
    # Validación cruzada: 5 folds, 10 repeticiones
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

    # Modelos a evaluar con parámetros ajustados
    modelos = {
        "Regresión Logística": LogisticRegression(max_iter=2000),  # Aumentamos el número de iteraciones
        "Naive Bayes": GaussianNB()
    }

    # Evaluar cada modelo
    for nombre, modelo in modelos.items():
        scores = cross_val_score(modelo, X, y, cv=cv, scoring='accuracy')
        print(f"{nombre}: Accuracy promedio = {scores.mean():.4f}, Desviación estándar = {scores.std():.4f}")
    
    print("-" * 50)

## DataSet: Spam

In [8]:
# Convertir etiquetas a arreglo plano por si acaso
y_spam_bin = y_spam.astype(int).values.ravel()

# Asignar datos escalados
X_spam_bin = X_spam_scaled

# Evaluación
evaluar_modelo(X_spam_bin, y_spam_bin, "Spam (binario)")

Evaluación en dataset: Spam (binario)
Regresión Logística: Accuracy promedio = 0.9702, Desviación estándar = 0.0048
Naive Bayes: Accuracy promedio = 0.9136, Desviación estándar = 0.0082
--------------------------------------------------


## DataSet: Cancer

In [9]:
# Etiquetas (y): se convierte a binario: 2 → 0, 4 → 1
y_cancer = data_cancer_imputed.iloc[:, -1].replace({2: 0, 4: 1}).astype(int).values.ravel()

# Características (X)
X_cancer = data_cancer_imputed.drop(columns=data_cancer_imputed.columns[-1])

# Escalado
scaler_cancer = StandardScaler()
X_cancer_scaled = scaler_cancer.fit_transform(X_cancer)

# Evaluación
evaluar_modelo(X_cancer_scaled, y_cancer, "Tumores de Seno")

Evaluación en dataset: Tumores de Seno
Regresión Logística: Accuracy promedio = 0.9642, Desviación estándar = 0.0129
Naive Bayes: Accuracy promedio = 0.9597, Desviación estándar = 0.0144
--------------------------------------------------


# **Regresión lineal a manita**

## ¿Qué es la regresión logística?

La **regresión logística** es un modelo de **clasificación binaria**. Su objetivo es predecir la probabilidad de que una instancia pertenezca a la clase positiva (por lo general, clase `1`).

Consta de los siguientes pasos importantes: 

---

### 1. Modelo matemático

La regresión logística modela la probabilidad de la clase positiva como:

$$
P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b)
$$

Donde:

- $\mathbf{x}$ es el vector de entrada (atributos).
- $\mathbf{w}$ es el vector de pesos (uno por cada atributo).
- $b$ es el término de sesgo (bias).
- $\sigma(z)$ es la función sigmoide, que convierte cualquier número real en un valor entre 0 y 1.

---

### 2. Función sigmoide

La **función sigmoide** se define como:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Convierte la salida lineal del modelo ($z = \mathbf{w}^\top \mathbf{x} + b$) en una **probabilidad**.

---

### 3. Función de pérdida (log-loss)

Para entrenar el modelo, se utiliza la **pérdida logarítmica** (también llamada entropía cruzada):

$$
\mathcal{L} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]
$$

Donde:

- $y^{(i)}$ es la etiqueta real (0 o 1).
- $\hat{y}^{(i)} = \sigma(\mathbf{w}^\top \mathbf{x}^{(i)} + b)$ es la predicción del modelo para la instancia $i$.

---

### 4. Entrenamiento con descenso de gradiente

Los pesos se ajustan para minimizar la función de pérdida, usando **gradientes**:

- Derivadas parciales para cada peso:

  $$
  \frac{\partial \mathcal{L}}{\partial w_j} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}^{(i)} - y^{(i)}) x_j^{(i)}
  $$

- Derivada para el bias:

  $$
  \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}^{(i)} - y^{(i)})
  $$

Los pesos se actualizan con:

```python
w -= learning_rate * grad_w
b -= learning_rate * grad_b


## Regresión Logística a manita, implementación

In [12]:
# Clase de Regresión Logística implementada a mano
class RegresionLogisticaManita:
    def __init__(self, learning_rate=0.01, n_iter=2000):
        # Inicializa los hiperparámetros de entrenamiento
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.w = None  # Pesos (vector de características)
        self.b = None  # Sesgo (bias)

    def sigmoide(self, z):
        # Función de activación sigmoide
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        # Entrena el modelo usando descenso de gradiente
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iter):
            # Predicción lineal
            z = np.dot(X, self.w) + self.b
            y_pred = self.sigmoide(z)

            # Gradientes para los pesos y el sesgo
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            # Actualización de los parámetros
            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

    def predecir_probabilidad(self, X):
        # Devuelve la probabilidad de clase positiva
        z = np.dot(X, self.w) + self.b
        return self.sigmoide(z)

    def predecir(self, X):
        # Devuelve la predicción final (0 o 1) usando un umbral de 0.5
        return (self.predecir_probabilidad(X) >= 0.5).astype(int)


# Función para evaluar el modelo implementado a mano
def evaluar_modelo_a_manita(X, y, nombre_dataset):
    """
    Entrena y evalúa un modelo de regresión logística implementado desde cero.
    
    Parámetros:
    -----------
    X : array-like
        Matriz de características (atributos de entrada), sin escalar.
    
    y : array-like
        Vector de etiquetas (0 o 1).
    
    nombre_dataset : str
        Nombre del conjunto de datos para impresión en pantalla.

    Salida:
    -------
    Imprime el accuracy del modelo entrenado en todo el conjunto
    y el promedio y desviación estándar del accuracy en validación cruzada (5 folds).
    """
    
    print(f"\n Evaluación para: {nombre_dataset}")

    # Escalar características (media 0, varianza 1)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Entrenamiento y evaluación en todo el conjunto (referencia rápida)
    modelo = RegresionLogisticaManita()
    modelo.fit(X_scaled, y)
    predicciones = modelo.predecir(X_scaled)
    acc_total = accuracy_score(y, predicciones)
    print(f" Accuracy sin validación: {acc_total:.4f}")

    # Evaluación con validación cruzada (5 particiones estratificadas)
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accuracies = []

    for train_idx, test_idx in kf.split(X_scaled, y):
        # Separar datos de entrenamiento y prueba
        X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Entrenar modelo en el fold actual
        modelo_cv = RegresionLogisticaManita()
        modelo_cv.fit(X_train, y_train)

        # Evaluar en el fold actual
        y_pred = modelo_cv.predecir(X_test)
        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)

    # Imprimir resultados finales
    print(f"Accuracy promedio (5 folds): {np.mean(accuracies):.4f}")
    print(f"Desviación estándar: {np.std(accuracies):.4f}")

## Aplicar la función al dataset: Spam

In [13]:
# --- Evaluar modelo implementado a mano en el dataset de spam ---
# No es necesario escalar aquí, porque la función ya lo hace internamente
# Asegurar que las etiquetas estén en formato adecuado
y_spam_bin = y_spam.astype(int).values.ravel()
X_spam_bin = X_spam  # Datos sin escalar

# Evaluar el modelo implementado a mano
evaluar_modelo_a_manita(X_spam_bin, y_spam_bin, "Spam (binario, modelo a mano)")


 Evaluación para: Spam (binario, modelo a mano)
 Accuracy sin validación: 0.9753
Accuracy promedio (5 folds): 0.9575
Desviación estándar: 0.0049


## Aplicar la función al dataset: Cancer

In [14]:
# Separar las características eliminando la última columna (que contiene las etiquetas)
X_cancer = data_cancer_imputed.drop(columns=data_cancer_imputed.columns[-1])

# Extraer la última columna como etiquetas (y),
# y mapear los valores: 2 → 0 , 4 → 1
# Luego se convierten a enteros y se aplana el arreglo
y_cancer = data_cancer_imputed.iloc[:, -1].replace({2: 0, 4: 1}).astype(int).values.ravel()

# Evaluar el modelo implementado a mano en el dataset de tumores de seno
evaluar_modelo_a_manita(X_cancer, y_cancer, "Tumores de Seno")


 Evaluación para: Tumores de Seno
 Accuracy sin validación: 0.9700
Accuracy promedio (5 folds): 0.9685
Desviación estándar: 0.0108


### ¿Por qué reetiquetar las clases en el dataset de cáncer?

El dataset de tumores de seno utilizaba las etiquetas `2` y `4` para representar las dos clases. Sin embargo, al implementar una regresión logística desde cero, es necesario que las etiquetas sean **binarias**, es decir, **0 y 1**.

---

### Problemas al usar etiquetas 2 y 4

1. **La función de pérdida no está diseñada para etiquetas distintas de 0 y 1**

   La función logarítmica usada en regresión logística es:

   $$
   \mathcal{L} = -\left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right]
   $$

   Si se usa `y = 2` o `y = 4`, esta fórmula produce resultados erróneos, e incluso puede generar **valores negativos**, lo cual no tiene sentido para una función de pérdida.

2. **La función sigmoide sólo produce valores entre 0 y 1**

   Como la salida del modelo es una probabilidad ($\hat{y}$), nunca podrá acercarse a `y = 4`, lo que provoca que la pérdida sea grande, el entrenamiento no tenga dirección útil y el modelo no aprenda nada.

3. **Las predicciones no son comparables con las etiquetas**

   Al final, comparamos las predicciones con las etiquetas reales para calcular el `accuracy`. Si las etiquetas son `2` y `4`, el modelo siempre se equivoca, aunque esté haciendo buenas predicciones de probabilidad.

# **DISCUSIÓN Y CONCLUSIONES**

### Evaluación del modelo implementado a manita

Además de comparar modelos con `scikit-learn`, se implementó una versión propia de **Regresión Logística desde cero** utilizando NumPy. Esta implementación fue evaluada en los mismos datasets con validación cruzada estratificada de 5 particiones.

### Resultados del modelo `RegresionLogisticaManita`

| Dataset            | Accuracy sin validación | Accuracy promedio (5 folds) | Desviación estándar |
|--------------------|-------------------------|------------------------------|----------------------|
| Cáncer             | 97.00%                  | 96.85%                       | 1.08%                |
| Spam (binario)     | 97.53%                  | 95.75%                       | 0.49%                |

Estos resultados son consistentes con los obtenidos con `scikit-learn`, especialmente en el caso de cáncer. En el caso del spam, aunque el accuracy sin validación fue alto, la validación cruzada reveló un desempeño más realista, aunque ligeramente por debajo del modelo de `scikit-learn`.

---

El modelo implementado a manita muestra que es posible alcanzar resultados bastante razonables respecto a librerías como `scikit-learn`. Si bien no incorpora optimizaciones como regularización, tolerancia o inicialización aleatoria, logra generalizar razonablemente bien en ambos datasets.

Por otro lado, **Naive Bayes**, aunque es un modelo simple y rápido, tuvo un desempeño notablemente menor en la clasificación de spam, probablemente debido a su fuerte suposición de independencia entre atributos, que no se cumple en ese caso.

En el caso del dataset de spam, **no fue necesario aplicar filtrado**, ya que desde el inicio solo contenía ejemplos de clases `0` y `1`. Esto facilitó el entrenamiento del modelo y la aplicación de validación cruzada estratificada sin problemas de clases poco representadas.

---

### Conclusión general

- Para ambas tareas, **Regresión Logística** (implementada a mano o con `sklearn`) fue superior tanto en precisión como en estabilidad.
- **Para cáncer**, ambos modelos son viables, pero se recomienda Regresión Logística por la ligera ventaja obtenida.
- **Para spam**, se recomienda Regresión Logística, ya que Naive Bayes no logró un rendimiento competitivo.